Deliverable 2 

Ida Sofie Ramsli & Emilie Marthinsen Kålås

-----------

Task 1

In [1]:
from pathlib import Path
import re


def read_log_file(file_path: Path) -> tuple[str, str]:
    """
    Read a log file and return its content and the encoding used
    """
    file_path = Path(file_path)

    try:
        # Try to read the file with utf-8 encoding
        with file_path.open("r", encoding="utf-8") as file:
            content = file.read()

        return content, "utf-8"
    
    except UnicodeDecodeError:
        # If utf-8 fails, try to read the file with latin-1 encoding
        with file_path.open("r", encoding="latin-1") as file:
            content = file.read()

        return content, "latin-1"



def parse_log_content(file_text_content: str) -> list[dict]:
    """
    Extract valid sales transaction records from the log file content
    """
    # Match the pattern of a valid sales transaction record
    pattern = (
        r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} "
        r"\d{2}:\d{2}:\d{2})\]\s+"
        r"\[(?P<register_id>[^\]]+)\]\s+"
        r"item=(?P<item>.+?)\s+"
        r"price=(?P<price>\d+(?:\.\d+)?)\s+"
        r"(?P<currency>[A-Z]{3})\s+"
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)"
    )



    transactions = []

    # Process each line in the log file content separately

    for line in file_text_content.splitlines():
        match = re.search(pattern, line)

        # Ignore lines that do not match the transaction record pattern
        if match:
            transaction = match.groupdict()

            # Convert the price from a string to a float
            transaction["price"] = float(transaction["price"])

            transactions.append(transaction)

    return transactions


The code firstly tries to read a given file with UFT-8 encoding. If it can't read the file using UFT-8, it then reads the file using Latin-1 encoding instead.

----------

Task 2

In [2]:
directory = Path("data")


def find_log_files(directory: Path) -> list[Path]:

    directory = Path(directory)

    # Create a list of all the log files
    log_files = []


    for item in directory.iterdir():
        if item.is_dir():
            # If the item is a directory, search inside it recursively
            log_files.extend(find_log_files(item))

        elif item.is_file() and item.suffix == ".log":
            # If the item is a log file, add it to the list
            log_files.append(item)


    return log_files



def find_active_registers(data) -> set[str]:

    # Create an empty set to store the unique register IDs
    active_registers = set()

    if isinstance(data, dict):

        if data.get("active") is True and isinstance(data.get("id"), str):
            # If the register is active, add its ID to the set
            active_registers.add(data["id"])

    # Search through all values in the dictionary recursively to find active registers
        for value in data.values():
            active_registers.update(find_active_registers(value))


    elif isinstance(data, list):
        # Search through all items in the list recursively to find active registers
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers


The first function, `find_log_files`, uses recursion in order to collect all the log files from the directory and return a list of all of them. The second function, `find_active_registers`, uses recursion to search the nested dictionaries and lists in the store configuration. It adds the ID's of all the active registers to a set and returns that set. 

-------------

Task 3

In [3]:
import numpy as np
import pandas as pd


def remove_price_outliers(prices) -> np.ndarray:
    """
    Finds the mean and sigma in the data, and replaces the outliers value with the median of the normal prices.
    Checks the assumption; when its over 3*sigma from the mean, is an outlier.  
    """
    arr = np.array(prices, dtype=float)
    mu = np.mean(arr)
    sigma = np.std(arr)

    is_outlier = np.abs(arr - mu) > 3 * sigma
    median_valid = np.median(arr[~is_outlier])

    cleaned = arr.copy()  
    cleaned[is_outlier] = median_valid
    return cleaned


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """ 
    Computes the refund ratio per item. Makes sure with 1e-6 
    that the items with 0 sold, does not crash the program 
    """
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    return refund_count / (sales_count + 1e-6)


In this task we cleaned the data for outliers, because it could give us a schewed result. We used the mean and standard deviation to find them, therafter we replaced them with the median to get a result as correct as possible. 


------

Task 4

In [4]:
def build_dataframe(records: list[dict]) -> pd.DataFrame:
    """ 
    Makes a Dataframe for the data, with the index being the timestamps
    """
    df = pd.DataFrame.from_records(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    df = df.sort_index()
    return df


def hourly_revenue_by_item(df: pd.DataFrame) -> pd.Series:
    """
    Groups the data by the item and, and resamples the transactions 
    in 1 -hour windows, and the price is calculated by the total price 
    """
    return df.groupby("item")["price"].resample("1h").sum()


def create_status_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates a pivot table that displays the total transaction revenue grouped by 
    item across status categories (Success, Refund, Cancelled)
    """
    pivot = df.pivot_table(
        values="price",
        index="item",
        columns="status",
        aggfunc="sum",
        fill_value=0.0,
    )
    for col in ("SUCCESS", "REFUND", "CANCELLED"):
        if col not in pivot.columns:
            pivot[col] = 0.0
    return pivot[["SUCCESS", "REFUND", "CANCELLED"]]


def compute_item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """
    Computing the refund ratio per item by using task 3 function
    """
    counts = (
        df.groupby(["item", "status"])
        .size()
        .unstack(fill_value=0)
    )

    for col in ("SUCCESS", "REFUND"):
        if col not in counts.columns:
            counts[col] = 0

    sales = counts["SUCCESS"].to_numpy(dtype=float)
    refunds = counts["REFUND"].to_numpy(dtype=float)
    ratios = compute_refund_ratio(sales, refunds)

    result = pd.DataFrame({
        "item": counts.index,
        "sales_count": sales.astype(int),
        "refund_count": refunds.astype(int),
        "refund_ratio": ratios,
    }).set_index("item")

    return result


In this task we converted the list of transaction to a Data frame. We used the timestaps as an index to sort the transactions. In the results we can see refund ratios, hourly revenue and revenue based of status.
Task 5



In [5]:
import json


def load_config(config_path: Path) -> dict:
    """
    Reading the JSON file, and turning it into a dictionaty
    """
    with open(config_path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_and_parse_logs(log_files: list[Path], active_registers: set[str]) -> list[dict]:
    all_records = []
    """
    Reading the logfiles, holding on to the transactions from active registers    
    """
    for log_path in log_files:
        content, encoding = read_log_file(log_path)
        records = parse_log_content(content)

        for rec in records:
            if rec["register_id"] in active_registers:
                all_records.append(rec)

    return all_records


def clean_prices(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replacing the outliers, with using function from task 3.
    """
    df = df.copy()
    df["price"] = remove_price_outliers(df["price"].to_numpy())
    return df


def apply_currency_correction(df: pd.DataFrame) -> pd.DataFrame:
    """
    Converting from NOK to EUR
    """
    df = df.copy()
    nok_mask = df["currency"] == "NOK"
    df.loc[nok_mask, "price"] = df.loc[nok_mask, "price"] * 10.0
    return df


def save_all_outputs(df: pd.DataFrame, hourly: pd.Series, pivot: pd.DataFrame,
                     ratios: pd.DataFrame, output_dir: Path) -> None:
    """
    Saving the four output files.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_dir / "cleaned_transactions.csv")

    
    hourly_dict = {}
    for (item, ts), value in hourly.items():
        if item not in hourly_dict:
            hourly_dict[item] = {}
        hourly_dict[item][ts.isoformat()] = float(value)

    with open(output_dir / "hourly_revenue.json", "w", encoding="utf-8") as f:
        json.dump(hourly_dict, f, indent=2)

    pivot.to_csv(output_dir / "status_pivot_table.csv")
    ratios.to_csv(output_dir / "item_refund_ratios.csv")


def run_workflow(data_dir: Path = Path("data"),
                 output_dir: Path = Path("output")) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.DataFrame]:
    """
    Running the workflow through the given data, creating a cleaned DataFrame, hourly revenue, status pivot table and refund ratios
    """
    log_files = find_log_files(data_dir)
    config = load_config(data_dir / "store_config.json")
    active_registers = find_active_registers(config)

    
    print(f"Found {len(log_files)} log files")
    print(f"Active registers: {sorted(active_registers)}")

    records = load_and_parse_logs(log_files, active_registers)
    print(f"Valid transactions after filtering: {len(records)}")

    
    df = build_dataframe(records)
    df = clean_prices(df)
    df = apply_currency_correction(df)

    
    hourly = hourly_revenue_by_item(df)
    pivot = create_status_pivot(df)
    ratios = compute_item_refund_ratios(df)

    
    save_all_outputs(df, hourly, pivot, ratios, output_dir)
    

    return df, hourly, pivot, ratios


df, hourly, pivot, ratios = run_workflow()

print("cleaned_transactions (first 5 rows)")
display(df.head())

print("status_pivot_table:")
display(pivot)

print("item_refund_ratios:")
display(ratios)

print("hourly_revenue:")
display(hourly.unstack(level=0))

Found 9 log files
Active registers: ['POS_01', 'POS_02', 'POS_03']
Valid transactions after filtering: 720
cleaned_transactions (first 5 rows)


,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:20,POS_03,Croissant,4.22,EUR,SUCCESS
2026-10-15 07:00:39,POS_01,Latte,4.15,EUR,SUCCESS
2026-10-15 07:01:49,POS_01,Cappuccino,41.60,NOK,SUCCESS
2026-10-15 07:02:02,POS_02,Muffin,33.50,NOK,CANCELLED
2026-10-15 07:02:13,POS_03,Sandwich,7.07,EUR,REFUND


status_pivot_table:


status,SUCCESS,REFUND,CANCELLED
item,,,
Cappuccino,1198.74,202.35,129.26
Croissant,1388.55,433.16,317.13
Espresso,653.76,182.53,373.83
Latte,1438.38,355.63,351.61
Muffin,1258.58,247.86,520.58
Sandwich,2239.40,633.35,412.83


item_refund_ratios:


,sales_count,refund_count,refund_ratio
item,,,
Cappuccino,66,9,0.136364
Croissant,102,22,0.215686
Espresso,70,17,0.242857
Latte,80,25,0.312500
Muffin,85,18,0.211765
Sandwich,76,22,0.289474


hourly_revenue:


item,Cappuccino,Croissant,Espresso,Latte,Muffin,Sandwich
timestamp,,,,,,
2026-10-15 07:00:00,792.69,693.44,497.83,947.85,902.64,1947.44
2026-10-15 08:00:00,561.32,1256.61,539.32,1124.96,1038.34,1156.47
2026-10-15 09:00:00,176.34,188.79,172.97,72.81,86.04,181.67


Connecting task 1 - 4, to one workflow that tidy up all the data. Its saves the data and saves it into four new files, saved in output, that can be used later. Each function has a responsibility, and can be reused and tested individually. 